# Blue team: train DeBERTa and evaluate a TF-IDF baseline
Run notebook 01 first. This notebook runs training only when you enable it below.
Default: FP32 model weights, batch 2, gradient accumulation 4, maximum length 256, early stopping, seed 42.
Pause the inference worker before training to avoid competition for the RTX 3070's 8 GiB VRAM:
`botfluencer-workstation --env-file ~/.config/botfluencer/runtime.env botfluencer blue pause`.
The service releases its resident model when paused. Resume after training.
Calibration uses validation only; test metrics are separate. Evaluate precision, false positives and confidence bounds rather than accuracy alone.


In [ ]:
from pathlib import Path

from botfluencer.blue.data import private_directory

# Copy this notebook into ~/botfluencer-notebooks before running it.
private_directory(Path.cwd())
DATA = Path("/mnt/d/botfluencer/data/blue-v1")
RELEASES = Path("/mnt/d/botfluencer/models/releases")
ACTIVE = Path("/mnt/d/botfluencer/models/blue-active")

In [ ]:
import torch

assert torch.cuda.is_available(), "Select the botfluencer CUDA kernel on gaming-pc"
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
from datetime import UTC, datetime

from botfluencer.blue.data import write_json
from botfluencer.blue.training import train

RUN_TRAINING = False  # Set True to start your training job.
if RUN_TRAINING:
    release = RELEASES / datetime.now(UTC).strftime("blue-%Y%m%dT%H%M%SZ")
    result = train(DATA, release, epochs=3, max_length=256)
    write_json(
        DATA / "last-release.json", {"path": str(release), "version": result["model_version"]}
    )
    print(result)
else:
    print("Training disabled; set RUN_TRAINING=True when the dataset is ready.")

## Interpret the result
Synthetic validation is a domain-shift diagnostic, not proof of real-player detection. Low false-positive counts on a small holdout are uncertain: inspect `precision_lower_95` and `fpr_upper_95`.
A threshold of 1.0 means no useful validation operating point was found; collect better data before activating it.
Compare DeBERTa with TF-IDF. If the simple baseline is as good or better, investigate lexical/topic leakage and improve data rather than assuming the larger model wins.
Before additional experiments, restart the notebook kernel to release GPU memory. Raw datasets, outputs, model releases and logs must stay outside OneDrive/Git.
